sourceID, verbatimScientificName (why not 100%)
year (which datasets are lacking)
datasetTitle (which sourceDatabase is lacking)
lifeStage and sex (how are they so high? is unknown being counted?)


In [ ]:
import pandas as pd
import os

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [ ]:
df = pd.read_parquet('Deduplicate/merged_all_deduplicated.parquet')

for col in ['sex', 'lifeStage']:
    if col in df.columns:
        uniques = pd.Series(df[col].dropna().unique()).sort_values().reset_index(drop=True)
        print(f"{col} unique entries:")
        print(uniques.to_string(index=False))
        print()
    else:
        print(f"Column not found: {col}")

In [ ]:
# summarize columns coverage for GBIF/east_data/occurrence.txt and save to GBIF/columns_east_occurrence.csv
infile = os.path.join('GBIF', 'east_data', 'occurrence.txt')
outcsv = os.path.join('GBIF', 'columns_east_occurrence.csv')

# get column names without loading full file
cols = pd.read_csv(infile, sep='\t', nrows=0).columns.tolist()
non_null_counts = {c: 0 for c in cols}
total = 0

# process in chunks to handle large files
for chunk in pd.read_csv(infile, sep='\t', dtype=str, chunksize=100000, low_memory=False):
    total += len(chunk)
    for c in cols:
        non_null_counts[c] += chunk[c].notna().sum()

# build summary dataframe
summary = pd.DataFrame({
    'column': cols,
    'non_null_count': [non_null_counts[c] for c in cols],
    'total_count': total
})
summary['coverage_percent'] = summary['non_null_count'] / summary['total_count'] * 100

# save and show top rows
summary.sort_values('coverage_percent', ascending=False).to_csv(outcsv, index=False)
print(f"Saved summary to: {outcsv}")
print(summary.sort_values('coverage_percent', ascending=False).head(50).to_string(index=False))